# Vecchia Lag-1 Neighbor Sensitivity with Godambe SE

Question: with the original grid resolution, if the current-time spatial neighborhood uses 15 nearest neighbors, how many same-neighborhood points should be used at `t-1`?

This notebook fixes:

- original grid resolution: `DELTA_LAT = 0.044`, `DELTA_LON = 0.063`
- current-time spatial neighbors: `limit_A = 15`
- previous-time lag: `t-1` only
- no `t-2` conditioning
- standard Vecchia kernel only

It compares:

```text
Lag1Nbs_03, Lag1Nbs_05, Lag1Nbs_07, Lag1Nbs_09, Lag1Nbs_11, Lag1Nbs_13, Lag1Nbs_15
```

In this kernel, `limit_B = k` means:

```text
t   : 15 current local spatial neighbors
t-1 : same location + k same-neighborhood locations
```

Instead of using `P90-P10` from many simulations, the notebook computes one-shot observed Godambe/sandwich standard errors from the fitted dataset:

```text
G = H^{-1} J H^{-1}
```

where `H` is the observed Hessian of the average Vecchia NLL and `J` is the outer product of per-unit conditional scores.


In [ ]:
import os
import sys
import time
import io
import contextlib
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.fft
import matplotlib.pyplot as plt

AMAREL_SRC = "/home/jl2815/tco"
LOCAL_SRC = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
_src = AMAREL_SRC if os.path.exists(AMAREL_SRC) else LOCAL_SRC
sys.path.insert(0, _src)

from GEMS_TCO import kernels_vecchia
from GEMS_TCO import orderings as _orderings

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float64

# Match the original GEMS target-grid resolution used in the local GIM script.
DELTA_LAT = 0.044
DELTA_LON = 0.063
T_STEPS = 8

print("DEVICE:", DEVICE)
print("SRC:", _src)
print("Grid resolution:", DELTA_LAT, DELTA_LON)


## Settings

`NUM_ITERS = 1` is the intended Godambe run. Increase it only if you also want a Monte Carlo check; Godambe itself is computed per fitted dataset.


In [ ]:
# Quick local defaults. Change these for the full experiment.
LAT_RANGE = (-1.0, 1.0)      # full: (-3.0, 2.0)
LON_RANGE = (123.0, 126.0)   # full: (121.0, 131.0)
NUM_ITERS = 1
SEED = 42

SMOOTH = 0.5
MM_COND_NUMBER = 100
NHEADS = 0

BASE_LIMIT_A = 15
LAG1_NEIGHBOR_COUNTS = [3, 5, 7, 9, 11, 13, 15]

# Disable Set C/t-2 by setting daily_stride beyond the number of simulated time steps.
DAILY_STRIDE = T_STEPS + 1

LBFGS_STEPS = 5
LBFGS_EVAL = 20
LBFGS_HIST = 10
INIT_NOISE = 0.7
SUPPRESS_FIT_PRINTS = True
COMPUTE_GODAMBE = True

HESSIAN_EPS = 1e-4
SCORE_EPS = 1e-5
H_RIDGE_SCALE = 1e-6

# Block/cluster sandwich J. This is closer to the standard dependent-data
# Godambe estimate than per-unit OPG because it preserves within-block score
# cross-products. Tune widths as a sensitivity check.
GODAMBE_J_METHOD = "block"
GODAMBE_BLOCK_LAT_WIDTH = 0.50
GODAMBE_BLOCK_LON_WIDTH = 0.50
GODAMBE_BLOCK_TIME_WIDTH = 2.0

TRUE_DICT = {
    "sigmasq": 10.0,
    "range_lat": 0.30,
    "range_lon": 0.40,
    "range_time": 2.0,
    "advec_lat": 0.08,
    "advec_lon": -0.126,
    "nugget": 2.5,
}

MODEL_SPECS = {
    f"Lag1Nbs_{k:02d}": {
        "lag1_nbs": k,
        "limit_A": BASE_LIMIT_A,
        "limit_B": k,
        "limit_C": 0,
        "description": f"t: 15 spatial nbs; t-1: same loc + {k} same-neighborhood nbs",
    }
    for k in LAG1_NEIGHBOR_COUNTS
}

pd.DataFrame(MODEL_SPECS).T


## Simulation and fitting helpers

All `lag1_nbs` values within an iteration use the same simulated field and the same initialization. This isolates the conditioning-set change.


In [ ]:
P_LABELS = ["sigmasq", "range_lat", "range_lon", "range_time", "advec_lat", "advec_lon", "nugget"]
P_COLS = ["sigmasq_est", "range_lat_est", "range_lon_est", "range_t_est", "advec_lat_est", "advec_lon_est", "nugget_est"]
SPATIAL_KEYS = ["sigmasq", "range_lat", "range_lon"]
ADVECTION_KEYS = ["advec_lat", "advec_lon"]

def transform_log_phi_to_physical(p):
    phi1, phi2, phi3, phi4 = (torch.exp(p[i]) for i in range(4))
    rlon = 1.0 / phi2
    return torch.stack([
        phi1 / phi2,
        rlon / torch.sqrt(phi3),
        rlon,
        rlon / torch.sqrt(phi4),
        p[4],
        p[5],
        torch.exp(p[6]),
    ])

def get_covariance_on_grid(lx, ly, lt, params):
    params = torch.clamp(params, min=-15.0, max=15.0)
    phi1, phi2, phi3, phi4 = (torch.exp(params[i]) for i in range(4))
    u_lat = lx - params[4] * lt
    u_lon = ly - params[5] * lt
    dist = torch.sqrt(u_lat.pow(2) * phi3 + u_lon.pow(2) + lt.pow(2) * phi4 + 1e-8)
    return (phi1 / phi2) * torch.exp(-dist * phi2)

def build_target_grid(lat_range, lon_range):
    lat0, lat1 = float(min(lat_range)), float(max(lat_range))
    lon0, lon1 = float(min(lon_range)), float(max(lon_range))
    n_lat = int(np.floor((lat1 - lat0) / DELTA_LAT + 1e-9)) + 1
    n_lon = int(np.floor((lon1 - lon0) / DELTA_LON + 1e-9)) + 1
    lats = lat0 + torch.arange(n_lat, device=DEVICE, dtype=DTYPE) * DELTA_LAT
    lons = lon0 + torch.arange(n_lon, device=DEVICE, dtype=DTYPE) * DELTA_LON
    lats = torch.round(lats * 10000) / 10000
    lons = torch.round(lons * 10000) / 10000
    g_lat, g_lon = torch.meshgrid(lats, lons, indexing="ij")
    grid_coords = torch.stack([g_lat.flatten(), g_lon.flatten()], dim=1)
    return lats, lons, grid_coords

def grid_resolution_report(lats, lons):
    lat_d = torch.diff(lats).detach().cpu().numpy()
    lon_d = torch.diff(lons).detach().cpu().numpy()
    return {
        "lat_min_step": float(lat_d.min()) if len(lat_d) else np.nan,
        "lat_max_step": float(lat_d.max()) if len(lat_d) else np.nan,
        "lon_min_step": float(lon_d.min()) if len(lon_d) else np.nan,
        "lon_max_step": float(lon_d.max()) if len(lon_d) else np.nan,
    }

def generate_field_values(n_lat, n_lon, t_steps, params):
    cpu = torch.device("cpu")
    f32 = torch.float32
    px, py, pt = 2 * n_lat, 2 * n_lon, 2 * t_steps
    lx = torch.arange(px, device=cpu, dtype=f32) * DELTA_LAT
    lx[px // 2:] -= px * DELTA_LAT
    ly = torch.arange(py, device=cpu, dtype=f32) * DELTA_LON
    ly[py // 2:] -= py * DELTA_LON
    lt = torch.arange(pt, device=cpu, dtype=f32)
    lt[pt // 2:] -= pt
    params_cpu = params.cpu().float()
    Lx, Ly, Lt = torch.meshgrid(lx, ly, lt, indexing="ij")
    C = get_covariance_on_grid(Lx, Ly, Lt, params_cpu)
    S = torch.fft.fftn(C)
    S.real = torch.clamp(S.real, min=0)
    noise = torch.fft.fftn(torch.randn(px, py, pt, device=cpu, dtype=f32))
    field = torch.fft.ifftn(torch.sqrt(S.real) * noise).real[:n_lat, :n_lon, :t_steps]
    return field.to(device=DEVICE, dtype=DTYPE)

def assemble_reg_map(field, grid_coords, true_params, t_offset=21.0):
    nugget_std = torch.sqrt(torch.exp(true_params[6]))
    n_grid = grid_coords.shape[0]
    field_flat = field.reshape(n_grid, field.shape[-1])
    reg_map = {}
    for t_idx in range(field.shape[-1]):
        key = f"t{t_idx}"
        t_val = float(t_offset + t_idx)
        dummy = torch.zeros(7, device=DEVICE, dtype=DTYPE)
        if t_idx > 0:
            dummy[t_idx - 1] = 1.0
        rows = torch.zeros((n_grid, 11), device=DEVICE, dtype=DTYPE)
        rows[:, :2] = grid_coords
        rows[:, 2] = field_flat[:, t_idx] + torch.randn(n_grid, device=DEVICE, dtype=DTYPE) * nugget_std
        rows[:, 3] = t_val
        rows[:, 4:] = dummy.unsqueeze(0).expand(n_grid, -1)
        reg_map[key] = rows.detach()
    return reg_map

def compute_grid_ordering(grid_coords, mm_cond_number):
    coords_np = grid_coords.detach().cpu().numpy()
    ord_mm = _orderings.maxmin_cpp(coords_np)
    nns = _orderings.find_nns_l2(locs=coords_np[ord_mm], max_nn=mm_cond_number)
    return ord_mm, nns

def true_to_log_params(true_dict):
    phi2 = 1.0 / true_dict["range_lon"]
    phi1 = true_dict["sigmasq"] * phi2
    phi3 = (true_dict["range_lon"] / true_dict["range_lat"]) ** 2
    phi4 = (true_dict["range_lon"] / true_dict["range_time"]) ** 2
    return [np.log(phi1), np.log(phi2), np.log(phi3), np.log(phi4), true_dict["advec_lat"], true_dict["advec_lon"], np.log(true_dict["nugget"])]

def backmap_params(out_params):
    p = [x.item() if isinstance(x, torch.Tensor) else float(x) for x in out_params[:7]]
    phi2 = np.exp(p[1])
    phi3 = np.exp(p[2])
    phi4 = np.exp(p[3])
    rlon = 1.0 / phi2
    return {
        "sigmasq": np.exp(p[0]) / phi2,
        "range_lat": rlon / phi3 ** 0.5,
        "range_lon": rlon,
        "range_time": rlon / phi4 ** 0.5,
        "advec_lat": p[4],
        "advec_lon": p[5],
        "nugget": np.exp(p[6]),
    }

def rmsre_for_keys(est, true_dict, keys, zero_thresh=0.01):
    vals = []
    for key in keys:
        tv = true_dict[key]
        ev = est[key]
        if abs(tv) >= zero_thresh:
            vals.append(((ev - tv) / abs(tv)) ** 2)
        else:
            vals.append(abs(ev - tv) ** 2)
    return float(np.sqrt(np.mean(vals)))

def relative_se_summary(se_by_key, denom_dict, keys, zero_thresh=0.01):
    vals = []
    for key in keys:
        denom = abs(denom_dict[key])
        if denom >= zero_thresh:
            vals.append((se_by_key[key] / denom) ** 2)
        else:
            vals.append(se_by_key[key] ** 2)
    return float(np.sqrt(np.mean(vals)))

def calculate_metrics(out_params, true_dict):
    est = backmap_params(out_params)
    return {
        "overall_rmsre": rmsre_for_keys(est, true_dict, P_LABELS),
        "spatial_rmsre": rmsre_for_keys(est, true_dict, SPATIAL_KEYS),
        "range_time_re": abs(est["range_time"] - true_dict["range_time"]) / abs(true_dict["range_time"]),
        "advec_rmsre": rmsre_for_keys(est, true_dict, ADVECTION_KEYS),
        "nugget_re": abs(est["nugget"] - true_dict["nugget"]) / abs(true_dict["nugget"]),
        "est": est,
    }

def make_random_init(rng, true_log, init_noise):
    noisy = list(true_log)
    for i in [0, 1, 2, 3, 6]:
        noisy[i] = true_log[i] + rng.uniform(-init_noise, init_noise)
    for i in [4, 5]:
        scale = max(abs(true_log[i]), 0.05)
        noisy[i] = true_log[i] + rng.uniform(-2 * scale, 2 * scale)
    return noisy


## Godambe helpers

This follows the observed-J approach in `sim_GIM_vecc_irr_dw_local_031926.py`: fix beta at the fitted GLS estimate, compute per-unit conditional NLL terms, then form score outer products by finite differences.


In [ ]:
def finite_diff_hessian(nll_fn, p, eps=HESSIAN_EPS):
    n = p.shape[0]
    H = torch.zeros(n, n, device=p.device, dtype=p.dtype)
    for i in range(n):
        p_p = p.detach().clone()
        p_m = p.detach().clone()
        p_p[i] += eps
        p_m[i] -= eps
        p_p.requires_grad_(True)
        p_m.requires_grad_(True)
        g_p = torch.autograd.grad(nll_fn(p_p), p_p)[0].detach()
        g_m = torch.autograd.grad(nll_fn(p_m), p_m)[0].detach()
        H[i] = (g_p - g_m) / (2.0 * eps)
    return (H + H.T) / 2.0

def vecchia_per_unit_target_coords(model):
    """Targets aligned with vecchia_per_unit_nll_terms: heads, then A/AB/ABC tails."""
    chunks = []
    if model.Heads_data is not None and model.Heads_data.shape[0] > 0:
        chunks.append(model.Heads_data[:, [0, 1, 3]].to(dtype=DTYPE))
    for X_b in [model.X_A, model.X_AB, model.X_ABC]:
        if X_b is not None and X_b.shape[0] > 0:
            chunks.append(X_b[:, -1, :].to(dtype=DTYPE))
    if not chunks:
        return torch.empty((0, 3), device=DEVICE, dtype=DTYPE)
    return torch.cat(chunks, dim=0)

def make_block_ids(target_coords):
    lat = target_coords[:, 0]
    lon = target_coords[:, 1]
    tim = target_coords[:, 2]

    lat_id = torch.floor((lat - lat.min()) / GODAMBE_BLOCK_LAT_WIDTH).to(torch.long)
    lon_id = torch.floor((lon - lon.min()) / GODAMBE_BLOCK_LON_WIDTH).to(torch.long)
    if GODAMBE_BLOCK_TIME_WIDTH is None or GODAMBE_BLOCK_TIME_WIDTH <= 0:
        time_id = torch.zeros_like(lat_id)
    else:
        time_id = torch.floor((tim - tim.min()) / GODAMBE_BLOCK_TIME_WIDTH).to(torch.long)

    n_lon = int(lon_id.max().item()) + 1 if lon_id.numel() else 1
    n_time = int(time_id.max().item()) + 1 if time_id.numel() else 1
    raw_id = (lat_id * n_lon + lon_id) * n_time + time_id
    _, block_id = torch.unique(raw_id, sorted=True, return_inverse=True)
    return block_id

def score_cov_per_unit_centered(score_mat):
    n_units = score_mat.shape[1]
    score_mean = score_mat.mean(dim=1)
    score_centered = score_mat - score_mean.unsqueeze(1)
    if n_units > 1:
        return score_centered @ score_centered.T / (n_units * (n_units - 1))
    return score_mat @ score_mat.T / max(n_units ** 2, 1)

def score_cov_block_cluster(score_mat, target_coords):
    """Cluster-robust Var(mean score), using block scores S_b = sum_{i in b} s_i.

    This keeps all within-block cross-products s_i s_j'. Cross-block covariance is
    still ignored, so block-size sensitivity is important.
    """
    n_units = score_mat.shape[1]
    scores = score_mat.T.contiguous()  # (N_units, p)
    block_id = make_block_ids(target_coords)
    n_blocks = int(block_id.max().item()) + 1 if block_id.numel() else 0
    block_scores = torch.zeros((n_blocks, scores.shape[1]), device=DEVICE, dtype=DTYPE)
    block_scores.index_add_(0, block_id, scores)

    if n_blocks > 1:
        centered = block_scores - block_scores.mean(dim=0, keepdim=True)
        J = centered.T @ centered * (n_blocks / (n_blocks - 1)) / (n_units ** 2)
    else:
        J = block_scores.T @ block_scores / max(n_units ** 2, 1)
    return J, n_blocks

def compute_vecchia_godambe(model, raw_params):
    p_hat = torch.tensor(raw_params[:7], device=DEVICE, dtype=DTYPE, requires_grad=True)

    def nll(p):
        return model.vecchia_batched_likelihood(p)

    H = finite_diff_hessian(nll, p_hat)
    eig = torch.linalg.eigvalsh(H).detach()
    h_abs_min = torch.clamp(torch.min(torch.abs(eig)), min=1e-12)
    h_cond = float((torch.max(torch.abs(eig)) / h_abs_min).detach().cpu())

    beta_hat = model.get_gls_beta(p_hat).detach()

    def per_unit_losses(p):
        return model.vecchia_per_unit_nll_terms(p, beta_hat)

    cols = []
    for k in range(p_hat.shape[0]):
        pp = p_hat.detach().clone()
        pm = p_hat.detach().clone()
        pp[k] += SCORE_EPS
        pm[k] -= SCORE_EPS
        with torch.no_grad():
            cols.append((per_unit_losses(pp) - per_unit_losses(pm)) / (2.0 * SCORE_EPS))
    score_mat = torch.stack(cols)  # (7, N_units), score_i = d ell_i / d theta
    n_units = score_mat.shape[1]

    target_coords = vecchia_per_unit_target_coords(model)
    if target_coords.shape[0] != n_units:
        raise RuntimeError(f"target/score mismatch: targets={target_coords.shape[0]}, scores={n_units}")

    score_mean = score_mat.mean(dim=1)
    p_grad = p_hat.detach().clone().requires_grad_(True)
    profile_grad = torch.autograd.grad(nll(p_grad), p_grad)[0].detach()
    score_grad_diff = profile_grad - score_mean

    J_uncentered = score_mat @ score_mat.T / (n_units ** 2)
    J_centered = score_cov_per_unit_centered(score_mat)
    J_block, n_blocks = score_cov_block_cluster(score_mat, target_coords)

    if GODAMBE_J_METHOD == "block":
        J_main = J_block
    elif GODAMBE_J_METHOD == "per_unit_centered":
        J_main = J_centered
    elif GODAMBE_J_METHOD == "per_unit_uncentered":
        J_main = J_uncentered
    else:
        raise ValueError(f"Unknown GODAMBE_J_METHOD={GODAMBE_J_METHOD!r}")

    eye = torch.eye(H.shape[0], device=DEVICE, dtype=DTYPE)
    h_scale = torch.clamp(torch.mean(torch.abs(torch.diag(H))), min=1.0)
    H_reg = H + eye * h_scale * H_RIDGE_SCALE
    H_inv = torch.linalg.pinv(H_reg)
    Jac = torch.autograd.functional.jacobian(transform_log_phi_to_physical, p_hat)

    def summarize_J(J):
        G_raw = H_inv @ J @ H_inv
        G_phys = Jac @ G_raw @ Jac.T
        se = torch.sqrt(torch.clamp(torch.diag(G_phys), min=0.0)).detach().cpu().numpy()
        se_by_key = dict(zip(P_LABELS, [float(x) for x in se]))
        return se_by_key, {
            "spatial": relative_se_summary(se_by_key, TRUE_DICT, SPATIAL_KEYS),
            "overall": relative_se_summary(se_by_key, TRUE_DICT, P_LABELS),
            "advec": relative_se_summary(se_by_key, TRUE_DICT, ADVECTION_KEYS),
            "nugget": se_by_key["nugget"] / abs(TRUE_DICT["nugget"]),
        }

    se_main, rel_main = summarize_J(J_main)
    se_block, rel_block = summarize_J(J_block)
    se_centered, rel_centered = summarize_J(J_centered)
    se_uncentered, rel_uncentered = summarize_J(J_uncentered)

    return {
        "gim_j_method": GODAMBE_J_METHOD,
        "gim_n_units": int(n_units),
        "gim_n_blocks": int(n_blocks),
        "gim_block_lat_width": float(GODAMBE_BLOCK_LAT_WIDTH),
        "gim_block_lon_width": float(GODAMBE_BLOCK_LON_WIDTH),
        "gim_block_time_width": float(GODAMBE_BLOCK_TIME_WIDTH),
        "gim_h_min_eig": float(eig.min().detach().cpu()),
        "gim_h_max_eig": float(eig.max().detach().cpu()),
        "gim_h_cond_abs": h_cond,
        "gim_score_mean_norm": float(torch.linalg.norm(score_mean).detach().cpu()),
        "gim_score_mean_max_abs": float(torch.max(torch.abs(score_mean)).detach().cpu()),
        "gim_profile_grad_norm": float(torch.linalg.norm(profile_grad).detach().cpu()),
        "gim_profile_grad_max_abs": float(torch.max(torch.abs(profile_grad)).detach().cpu()),
        "gim_score_profile_diff_norm": float(torch.linalg.norm(score_grad_diff).detach().cpu()),
        "gim_score_profile_diff_max_abs": float(torch.max(torch.abs(score_grad_diff)).detach().cpu()),
        "gim_spatial_rel_se": rel_main["spatial"],
        "gim_overall_rel_se": rel_main["overall"],
        "gim_advec_rel_se": rel_main["advec"],
        "gim_nugget_rel_se": rel_main["nugget"],
        "gim_spatial_rel_se_block": rel_block["spatial"],
        "gim_overall_rel_se_block": rel_block["overall"],
        "gim_advec_rel_se_block": rel_block["advec"],
        "gim_nugget_rel_se_block": rel_block["nugget"],
        "gim_spatial_rel_se_perunit_centered": rel_centered["spatial"],
        "gim_overall_rel_se_perunit_centered": rel_centered["overall"],
        "gim_advec_rel_se_perunit_centered": rel_centered["advec"],
        "gim_nugget_rel_se_perunit_centered": rel_centered["nugget"],
        "gim_spatial_rel_se_uncentered": rel_uncentered["spatial"],
        "gim_overall_rel_se_uncentered": rel_uncentered["overall"],
        "gim_advec_rel_se_uncentered": rel_uncentered["advec"],
        "gim_nugget_rel_se_uncentered": rel_uncentered["nugget"],
        **{f"gim_se_{k}": v for k, v in se_main.items()},
        **{f"gim_se_{k}_block": v for k, v in se_block.items()},
        **{f"gim_se_{k}_perunit_centered": v for k, v in se_centered.items()},
        **{f"gim_se_{k}_uncentered": v for k, v in se_uncentered.items()},
    }


In [ ]:
def fit_vecchia_spec(model_name, spec, reg_map_ord, nns_grid, initial_vals):
    params = [torch.tensor([val], device=DEVICE, dtype=DTYPE, requires_grad=True) for val in initial_vals]
    model = kernels_vecchia.fit_vecchia_lbfgs(
        smooth=SMOOTH,
        input_map=reg_map_ord,
        nns_map=nns_grid,
        mm_cond_number=MM_COND_NUMBER,
        nheads=NHEADS,
        limit_A=spec["limit_A"],
        limit_B=spec["limit_B"],
        limit_C=spec["limit_C"],
        daily_stride=DAILY_STRIDE,
    )

    t0 = time.time()
    if SUPPRESS_FIT_PRINTS:
        with contextlib.redirect_stdout(io.StringIO()):
            model.precompute_conditioning_sets()
    else:
        model.precompute_conditioning_sets()
    precompute_s = time.time() - t0

    optimizer = model.set_optimizer(params, lr=1.0, max_iter=LBFGS_EVAL, history_size=LBFGS_HIST)
    t1 = time.time()
    if SUPPRESS_FIT_PRINTS:
        with contextlib.redirect_stdout(io.StringIO()):
            out, n_iter = model.fit_vecc_lbfgs(params, optimizer, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    else:
        out, n_iter = model.fit_vecc_lbfgs(params, optimizer, max_steps=LBFGS_STEPS, grad_tol=1e-5)
    fit_s = time.time() - t1

    raw_params = [float(x) for x in out[:7]]
    metrics = calculate_metrics(out, TRUE_DICT)

    godambe = {}
    gim_s = 0.0
    if COMPUTE_GODAMBE:
        t2 = time.time()
        godambe = compute_vecchia_godambe(model, raw_params)
        gim_s = time.time() - t2

    return out, float(out[-1]), int(n_iter), precompute_s, fit_s, gim_s, metrics, godambe

def run_experiment(num_iters=NUM_ITERS, seed=SEED, save_csv=True):
    rng = np.random.default_rng(seed)
    torch.manual_seed(seed)
    true_log = true_to_log_params(TRUE_DICT)
    true_params = torch.tensor(true_log, device=DEVICE, dtype=DTYPE)

    lats_grid, lons_grid, grid_coords = build_target_grid(LAT_RANGE, LON_RANGE)
    n_lat, n_lon = len(lats_grid), len(lons_grid)
    print(f"Grid: {n_lat} x {n_lon} x {T_STEPS} = {n_lat * n_lon * T_STEPS:,} rows")
    print("Actual resolution:", grid_resolution_report(lats_grid, lons_grid))
    print(f"Models: A={BASE_LIMIT_A}, B in {LAG1_NEIGHBOR_COUNTS}, no Set C/t-2")

    ord_grid, nns_grid = compute_grid_ordering(grid_coords, MM_COND_NUMBER)
    print("Ordering done")

    records = []
    for it in range(num_iters):
        print(f"\n=== Iteration {it + 1}/{num_iters} ===")
        initial_vals = make_random_init(rng, true_log, INIT_NOISE)
        field = generate_field_values(n_lat, n_lon, T_STEPS, true_params)
        reg_map = assemble_reg_map(field, grid_coords, true_params)
        del field
        reg_map_ord = {k: v[ord_grid] for k, v in reg_map.items()}

        for model_name, spec in MODEL_SPECS.items():
            try:
                print(f"B={spec['lag1_nbs']:02d}: fitting", end="")
                out, loss, n_iter, pre_s, fit_s, gim_s, metrics, godambe = fit_vecchia_spec(
                    model_name, spec, reg_map_ord, nns_grid, initial_vals
                )
                est = metrics.pop("est")
                row = {
                    "iter": it + 1,
                    "model": model_name,
                    "lag1_nbs": spec["lag1_nbs"],
                    "loss": round(loss, 6),
                    "overall_rmsre": round(metrics["overall_rmsre"], 6),
                    "spatial_rmsre": round(metrics["spatial_rmsre"], 6),
                    "range_time_re": round(metrics["range_time_re"], 6),
                    "advec_rmsre": round(metrics["advec_rmsre"], 6),
                    "nugget_re": round(metrics["nugget_re"], 6),
                    "precompute_s": round(pre_s, 3),
                    "fit_s": round(fit_s, 3),
                    "gim_s": round(gim_s, 3),
                    "total_s": round(pre_s + fit_s + gim_s, 3),
                    "fit_iter": n_iter,
                }
                row.update({
                    "sigmasq_est": round(est["sigmasq"], 6),
                    "range_lat_est": round(est["range_lat"], 6),
                    "range_lon_est": round(est["range_lon"], 6),
                    "range_t_est": round(est["range_time"], 6),
                    "advec_lat_est": round(est["advec_lat"], 6),
                    "advec_lon_est": round(est["advec_lon"], 6),
                    "nugget_est": round(est["nugget"], 6),
                })
                row.update({k: round(v, 8) if isinstance(v, float) else v for k, v in godambe.items()})
                records.append(row)
                print(
                    f" | loss={loss:.4f} spatial_err={row['spatial_rmsre']:.4f} "
                    f"gim_spatial_se={row.get('gim_spatial_rel_se', np.nan):.4f} "
                    f"time={row['total_s']:.1f}s"
                )
            except Exception as e:
                print(f" | SKIP {model_name}: {type(e).__name__}: {e}")

    df = pd.DataFrame(records)
    if save_csv and len(df):
        out_dir = Path("log")
        out_dir.mkdir(exist_ok=True)
        out_path = out_dir / "sim_vecchia_lag1_neighbor_sensitivity_godambe_042926_results.csv"
        df.to_csv(out_path, index=False)
        print("Saved:", out_path)
    return df


## Run

This is intended as a one-shot Godambe comparison. The fitted error columns are still shown because the data are simulated, but the primary one-shot stability metric is `gim_spatial_rel_se`.


In [ ]:
df_results = run_experiment()
df_results.head()


## Main comparison

Lower `gim_spatial_rel_se` means the fitted design gives smaller sandwich standard error for the spatial covariance parameters relative to the true parameter scale.


In [ ]:
main_cols = [
    "lag1_nbs", "loss", "spatial_rmsre", "overall_rmsre",
    "gim_j_method", "gim_spatial_rel_se", "gim_spatial_rel_se_block",
    "gim_spatial_rel_se_perunit_centered", "gim_spatial_rel_se_uncentered",
    "gim_overall_rel_se", "gim_advec_rel_se", "gim_nugget_rel_se",
    "gim_score_mean_max_abs", "gim_profile_grad_max_abs", "gim_score_profile_diff_max_abs",
    "gim_h_cond_abs", "gim_n_units", "gim_n_blocks", "total_s",
]
summary = df_results[main_cols].sort_values("gim_spatial_rel_se").reset_index(drop=True)
summary


In [ ]:
param_cols = [
    "lag1_nbs",
    "sigmasq_est", "range_lat_est", "range_lon_est", "range_t_est", "advec_lat_est", "advec_lon_est", "nugget_est",
    "gim_se_sigmasq", "gim_se_range_lat", "gim_se_range_lon", "gim_se_range_time", "gim_se_advec_lat", "gim_se_advec_lon", "gim_se_nugget",
]
df_results[param_cols].sort_values("lag1_nbs")


In [ ]:
plot_df = df_results.sort_values("lag1_nbs")
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(plot_df["lag1_nbs"], plot_df["gim_spatial_rel_se"], marker="o", label="Godambe spatial rel SE")
axes[0].plot(plot_df["lag1_nbs"], plot_df["spatial_rmsre"], marker="s", label="realized spatial RMSRE")
axes[0].set_title("Spatial parameter stability")
axes[0].set_xlabel("t-1 neighbor count")
axes[0].legend()

axes[1].plot(plot_df["lag1_nbs"], plot_df["loss"], marker="o")
axes[1].set_title("Vecchia loss")
axes[1].set_xlabel("t-1 neighbor count")
axes[1].set_ylabel("loss")

axes[2].plot(plot_df["lag1_nbs"], plot_df["gim_overall_rel_se"], marker="o", label="overall")
axes[2].plot(plot_df["lag1_nbs"], plot_df["gim_advec_rel_se"], marker="s", label="advec")
axes[2].plot(plot_df["lag1_nbs"], plot_df["gim_nugget_rel_se"], marker="^", label="nugget")
axes[2].set_title("Godambe relative SE")
axes[2].set_xlabel("t-1 neighbor count")
axes[2].legend()

plt.tight_layout()
plt.show()


## Interpretation

- If `gim_spatial_rel_se` is minimized around a small or middle value, then the lagged local-neighborhood information saturates before using all 15 points.
- If `gim_spatial_rel_se` keeps decreasing through `15`, then the full lag-1 same-neighborhood set is still useful for spatial parameter identification.
- If `loss` improves but `gim_spatial_rel_se` worsens, the larger conditioning set improves fitted likelihood but not spatial-parameter stability.
- Godambe SE is a one-shot asymptotic uncertainty diagnostic. It does not measure Monte Carlo bias by itself, so it complements rather than fully replaces repeated simulation.
- `gim_spatial_rel_se` now uses a centered score covariance estimate; the old uncentered OPG version is kept as `gim_spatial_rel_se_uncentered`.
- Large `gim_score_profile_diff_max_abs` means the fixed-beta per-unit score is not matching the profiled objective well, so the Godambe result should be treated as diagnostic rather than definitive.
- `GODAMBE_J_METHOD="block"` uses cluster/block scores, which is closer to the standard dependent-data sandwich estimator than per-unit OPG.
- Block J preserves within-block score cross-products; per-unit J effectively treats conditional score terms as independent after centering.
- The block result should be checked under a few block widths because cross-block dependence is still ignored.


## Monte Carlo check under the same A=15 design

The Godambe section above is a one-shot asymptotic uncertainty diagnostic. It can disagree with realized simulation error because it measures local curvature/score variability around one fitted dataset, not repeated-sample bias or optimizer variability.

This section reruns the same conditioning designs over multiple simulated datasets with `COMPUTE_GODAMBE = False`, so it is much faster per replicate and directly estimates realized RMSRE distributions.


In [ ]:
# Monte Carlo settings. Increase for the final check.
MC_NUM_ITERS = 20       # full: 100 or 300
MC_SEED = 2026
MC_COMPUTE_GODAMBE = False

# Keep the same A=15, B in [3,5,7,9,11,13,15] model set from above.
print("MC model specs:")
pd.DataFrame(MODEL_SPECS).T[["lag1_nbs", "limit_A", "limit_B", "limit_C", "description"]]


In [ ]:
def run_monte_carlo_check(num_iters=MC_NUM_ITERS, seed=MC_SEED, save_csv=True):
    global COMPUTE_GODAMBE
    old_compute_godambe = COMPUTE_GODAMBE
    COMPUTE_GODAMBE = MC_COMPUTE_GODAMBE
    try:
        df_mc = run_experiment(num_iters=num_iters, seed=seed, save_csv=False)
    finally:
        COMPUTE_GODAMBE = old_compute_godambe

    if save_csv and len(df_mc):
        out_dir = Path("log")
        out_dir.mkdir(exist_ok=True)
        out_path = out_dir / "sim_vecchia_lag1_neighbor_sensitivity_mc_042926_results.csv"
        df_mc.to_csv(out_path, index=False)
        print("Saved:", out_path)
    return df_mc

df_mc = run_monte_carlo_check()
df_mc.head()


In [ ]:
def mc_p90_p10(x):
    return np.percentile(x, 90) - np.percentile(x, 10)

def summarize_mc(df):
    metric_cols = ["loss", "overall_rmsre", "spatial_rmsre", "range_time_re", "advec_rmsre", "nugget_re", "total_s"]
    rows = []
    for lag1_nbs, sub in df.groupby("lag1_nbs"):
        row = {"lag1_nbs": lag1_nbs, "n": len(sub)}
        for col in metric_cols:
            vals = sub[col].dropna().to_numpy()
            row[f"{col}_mean"] = np.mean(vals)
            row[f"{col}_median"] = np.median(vals)
            row[f"{col}_p90_p10"] = mc_p90_p10(vals)
        rows.append(row)
    return pd.DataFrame(rows).sort_values("lag1_nbs").reset_index(drop=True)

mc_summary = summarize_mc(df_mc)
mc_summary.sort_values("spatial_rmsre_mean")


In [ ]:
display(mc_summary)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(mc_summary["lag1_nbs"], mc_summary["spatial_rmsre_mean"], marker="o", label="MC mean")
axes[0].plot(mc_summary["lag1_nbs"], mc_summary["spatial_rmsre_median"], marker="s", label="MC median")
if "summary" in globals() and "gim_spatial_rel_se" in summary.columns:
    godambe_plot = summary.sort_values("lag1_nbs")
    axes[0].plot(godambe_plot["lag1_nbs"], godambe_plot["gim_spatial_rel_se"], marker="^", label=f"Godambe rel SE ({GODAMBE_J_METHOD})")
axes[0].set_title("Spatial parameter stability")
axes[0].set_xlabel("t-1 neighbor count")
axes[0].legend()

axes[1].plot(mc_summary["lag1_nbs"], mc_summary["overall_rmsre_mean"], marker="o", label="overall")
axes[1].plot(mc_summary["lag1_nbs"], mc_summary["advec_rmsre_mean"], marker="s", label="advec")
axes[1].plot(mc_summary["lag1_nbs"], mc_summary["nugget_re_mean"], marker="^", label="nugget")
axes[1].set_title("Monte Carlo realized error")
axes[1].set_xlabel("t-1 neighbor count")
axes[1].legend()

axes[2].plot(mc_summary["lag1_nbs"], mc_summary["loss_mean"], marker="o", label="MC loss")
axes[2].set_title("Monte Carlo Vecchia loss")
axes[2].set_xlabel("t-1 neighbor count")
axes[2].set_ylabel("loss")

plt.tight_layout()
plt.show()


## Why Godambe and Monte Carlo can disagree

- Godambe SE is local: it evaluates curvature and score variability around one fitted dataset.
- Monte Carlo RMSRE is repeated-sample realized error: it includes finite-sample bias, random-field variability, and optimizer sensitivity.
- The earlier small simulation used `A=8`, `B=0..8`, and only a few replicates; this notebook uses `A=15`, `B in {3,5,7,9,11,13,15}`.
- Therefore, disagreement is not necessarily a bug. If both Godambe and Monte Carlo favor the same `B`, that is strong evidence. If not, report the distinction: Godambe favors local identifiability, while Monte Carlo shows realized finite-sample recovery.
